# `ai_parse_document` — Old article preprocess extraction

## Overview
This notebook benchmarks Databricks `ai_parse_document` (v2.0) :

| File | Format | Key challenge |
|---|---|---|
| `old_articles.pdf` | Scanned PDF | bad alignement / low-resolution OCR |


In the last notebook 'ai_parse_document_debug', the function failed to extract the content of the PDF of an old newspaper article.
![Article preview](https://raw.githubusercontent.com/O-Faraday/databricks_genai_demo/main/data/multiformat/old_article_presentation.png)



In [0]:
%pip install -q pymupdf Pillow numpy landingai-ade --quiet
# PyMuPDF (fitz) replaces pdf2image: pure Python, no poppler binary required
dbutils.library.restartPython()


### Config Unity Catalog

In [0]:
%run ../_config/config_unity_catalog

### Load the debug function

In [0]:
%run ./document_renderer

## 1. Set up

In [0]:
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Paths ──────────────────────────────────────────────────────────────────
PATH_VOLUME   = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_RESULTS  = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"

# Create output directories if they don't exist
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Files to analyse ───────────────────────────────────────────────────────
FILES = {
    "old_articles"  : "old_articles.pdf",
}

print(f"Volume path   : {PATH_VOLUME}")
print(f"Results output: {PATH_RESULTS}")
print(f"Files to parse: {list(FILES.values())}")

## 2. Parse Document Function with `ai_parse_document` 

We use:
- `version = '2.0'` — latest schema (Jan 2026), based on `elements` array (replaces the legacy `pages` array)
- `descriptionElementTypes = '*'` — in v2.0, `'*'` and `'figure'` produce **the same behaviour**: AI descriptions are generated for **figures only** (descriptions for tables and text blocks are not yet supported)
- `imageOutputPath` — persist rendered page images to the volume for visual inspection or multi-modal RAG

In [0]:
def parse_doc(source, output) : 
  # SQL statement with ai_parse_document()
  sql = f'''
  with parsed_documents AS (
    SELECT
      path,
      ai_parse_document(content
       ,
      map(
      'version', '2.0',
       'imageOutputPath', '{output}',
       'descriptionElementTypes', 'figure' 
      )
    ) as parsed
  FROM
    read_files('{source}', format => 'binaryFile')
  )
  select * from parsed_documents
  '''

  parsed_results = [row.parsed for row in spark.sql(sql).collect()]
  return parsed_results

## 3. Preprocess Scanned Document Deep Dive -- `old_articles.pdf`

The baseline parse returned **0 elements** of the notebook "ai_parse_document_debug" because `old_articles.pdf` is a 1908 French journal scanned at very low resolution. This section applies a preprocessing pipeline to rescue the document before re-parsing.

### 6a. PIL Preprocessing Pipeline

Three steps applied per page:

| Step | Technique | Purpose |
|---|---|---|
| **Upscale** | PIL `LANCZOS` resize to 300 DPI equivalent | More pixels for OCR |
| **Binarise** | Otsu global threshold via numpy | Removes grey noise, sharp B&W |
| **Deskew** | Projection-profile angle scan | Corrects tilt from scanning |

The preprocessed pages are re-assembled into a new PDF and saved to the volume.

In [0]:
import io, time
import numpy as np
import fitz                              # PyMuPDF — no poppler required
from PIL import Image, ImageFilter

SOURCE_PDF       = f"{PATH_VOLUME}/{FILES['old_articles']}"
PREPROCESSED_PDF = f"{PATH_VOLUME}/preprocessed_{FILES['old_articles']}"
TARGET_DPI       = 300
ZOOM             = TARGET_DPI / 72.0     # fitz renders at 72 dpi by default

pdf_doc = fitz.open(SOURCE_PDF)
print(f"Source PDF : {SOURCE_PDF}")
print(f"Pages      : {pdf_doc.page_count}  |  target DPI: {TARGET_DPI}")


def deskew(pil_img, angle_range=5.0, steps=100):
    """Projection-profile deskew: maximise row-sum variance to find skew angle."""
    gray = np.array(pil_img.convert('L'))
    best_angle, best_score = 0.0, -1.0
    for angle in np.linspace(-angle_range, angle_range, steps):
        rotated  = Image.fromarray(gray).rotate(angle, expand=False, fillcolor=255)
        row_sums = np.array(rotated).sum(axis=1).astype(float)
        score    = float(np.var(row_sums))
        if score > best_score:
            best_score, best_angle = score, angle
    if abs(best_angle) > 0.1:
        return pil_img.rotate(best_angle, expand=False, fillcolor=255,
                               resample=Image.BICUBIC)
    return pil_img


def binarise(pil_img):
    """Otsu global threshold binarisation (pure numpy, no OpenCV)."""
    gray    = np.array(pil_img.convert('L'))
    hist, _ = np.histogram(gray.flatten(), 256, [0, 256])
    hist    = hist.astype(float)
    total   = gray.size
    sum_all = float((np.arange(256) * hist).sum())
    wb = sb  = 0.0
    best_thresh, best_var = 0, 0.0
    for t in range(256):
        wb += hist[t]; wf = total - wb
        if wb == 0 or wf == 0: continue
        sb += t * hist[t]
        mb, mf = sb / wb, (sum_all - sb) / wf
        var = wb * wf * (mb - mf) ** 2
        if var > best_var:
            best_var, best_thresh = var, t
    return Image.fromarray((gray > best_thresh).astype('uint8') * 255).convert('RGB')


# ── Rasterise + preprocess each page with fitz ─────────────────────────────
processed_pages = []
mat = fitz.Matrix(ZOOM, ZOOM)            # scale matrix for target DPI

for page_num in range(pdf_doc.page_count):
    t0   = time.time()
    page = pdf_doc.load_page(page_num)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)

    # fitz pixmap -> PIL Image
    raw_img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    w0, h0  = raw_img.size

    # Step 1: sharpen to recover edge detail lost in scan
    sharpened = raw_img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=120, threshold=3))
    # Step 2: binarise (Otsu)
    binary    = binarise(sharpened)
    # Step 3: deskew
    deskewed  = deskew(binary)

    processed_pages.append(deskewed)
    print(f"  Page {page_num+1}: {w0}x{h0}px -> preprocessed in {round(time.time()-t0,2)}s")

pdf_doc.close()

# ── Re-assemble into PDF ────────────────────────────────────────────────────
processed_pages[0].save(
    PREPROCESSED_PDF, save_all=True,
    append_images=processed_pages[1:],
    format='PDF', resolution=TARGET_DPI,
)
import os
print(f"\nPreprocessed PDF saved : {PREPROCESSED_PDF}")
print(f"Size                   : {os.path.getsize(PREPROCESSED_PDF)//1024} KB")
print(f"Pages                  : {len(processed_pages)}")


### 2b. Debug Preprocessed PDF with `ai_parse_document`

Re-run `ai_parse_document v2.0` on the preprocessed PDF and compare the scorecard against the raw baseline.

In [0]:
parsed_results = parse_doc(PREPROCESSED_PDF, PATH_RESULTS)
render_ai_parse_output_interactive(parsed_results)
